# ODR API Client Test Notebook

This notebook demonstrates all the functionality of the ODR (Open Data Repository) API client. Make sure the `ODR_API_Client.py` script is in the same directory as this notebook.

### 🚧 *Further Features & Functionality Incoming...*
For any questions, suggestions, or feedback, feel free to reach out to:

**Abdullah Shahid**  
📩 `amshahid003 [at] gmail.com`  
🎓 `amshahid [at] unc.edu` or `amshahid [at] ncsu.edu`

## 1. Setup and Import

In [2]:
!pip install jwt

  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached pycparser-2.23-py3-none-any.whl.metadata (993 bytes)
   ---------------------------------------- 0.0/3.5 MB ? eta -:--:--
   -- ------------------------------------- 0.3/3.5 MB ? eta -:--:--
   ----------- ---------------------------- 1.0/3.5 MB 3.1 MB/s eta 0:00:01
   ----------------- ---------------------- 1.6/3.5 MB 3.0 MB/s eta 0:00:01
   ----------------------------- ---------- 2.6/3.5 MB 3.6 MB/s eta 0:00:01
   ---------------------------------------- 3.5/3.5 MB 4.1 MB/s eta 0:00:00
Using cached typing_extensions-4.15.0-py3-none-any.whl (44 kB)
Using cached pycparser-2.23-py3-none-any.whl (118 kB)

  Attempting uninstall: typing-extensions

    Found existing installation: typing_extensions 4.12.2

    Uninstalling typing_extensions-4.12.2:

   ---------------------------------------- 0/5 [typing-extensions]
   ---------------------------------------- 0/5 [typing-extensions]
   ------------

In [7]:
# Import required libraries
import json
import os
from pathlib import Path
import time

# Import the ODR API client from your script
from ODR_API_Client import ODRAPIClient

# Configuration
BASE_URL = "https://www.odr.io/api/v4"
USERNAME = "amshahid@ncsu.edu"
PASSWORD = "qkh8fjd6adh*NPU!ekn"  # Note: In production, use environment variables for credentials
DATASET_UUID = "063c0d3d4bd183ab0dda87c544ae"

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


In [15]:
# =============================================================================
# View All Fields in a Record
# =============================================================================
# Fetch a record and display all its fields with details

record_uuid = "420a340058dd57b5b162536b6f33"  # Change to any record UUID

# Get the record
record = client.get_record(record_uuid)

print(f"Record: {record.get('record_name')} ({record_uuid})")
print("=" * 70)

for field in record.get("fields", []):
    field_name = field.get("field_name", "Unknown")
    field_uuid = field.get("field_uuid", "")
    template_uuid = field.get("template_field_uuid", "")
    
    print(f"\n📌 {field_name}")
    print(f"   field_uuid: {field_uuid}")
    print(f"   template_field_uuid: {template_uuid}")
    
    # Check for different value types
    if "value" in field:
        print(f"   value: {field['value']}")
    elif "values" in field:
        for v in field.get("values", []):
            selected = "✓" if v.get("selected") else " "
            print(f"   [{selected}] {v.get('name')} (option_uuid: {v.get('template_radio_option_uuid', 'N/A')})")
    elif "files" in field:
        for f in field.get("files", []):
            print(f"   📄 {f.get('file_name')} (uuid: {f.get('file_uuid', 'N/A')})")

Fetching record: 420a340058dd57b5b162536b6f33
Record: 777390 (420a340058dd57b5b162536b6f33)

📌 Data File
   field_uuid: a65467babf8a1ac7e1d7319e3928
   template_field_uuid: None
   📄 None (uuid: e777063959042f86e1f7543df8cf)

📌 Data Type
   field_uuid: 996f2f04be5e12bc6d251e54bb8f
   template_field_uuid: None
   [✓] Elemental (option_uuid: 739c6bd78291c461b25523568c39)

📌 Source ID
   field_uuid: 98c0dc4db715d503abc93fa598f9
   template_field_uuid: None
   value: Dick_1992

📌 Class
   field_uuid: 676b2e7658da32d4c518b3877401
   template_field_uuid: None
   [✓] Non-Indicative (option_uuid: 8ef863b49d28db481480a9053044)

📌 Subclass
   field_uuid: bcf6ab5a9b02de9e0594772f2c2a
   template_field_uuid: None
   [✓] Non-Indicative (option_uuid: ace0048aa804b5ce1a7bc6829a13)

📌 Sample Type
   field_uuid: 423044bee60c5e83fcb7fbf1b713
   template_field_uuid: None
   [✓] basalt (option_uuid: 7401f60d37a3400259c2864ece78)

📌 Source Links
   field_uuid: cb24ce292d861629416b51c40aa0
   template_field

## 2. Initialize Client and Authenticate

The first step is to create an API client instance and authenticate to get a Bearer to

In [8]:
# Initialize the API client
client = ODRAPIClient(BASE_URL, USERNAME, PASSWORD)

# Authenticate and get token
try:
    token = client.authenticate()
    print(f"✅ Authentication successful!")
    print(f"🔑 Token (first 20 chars): {token[:20]}...")
    print(f"📅 Token expiry: {client.token_expiry}")
except Exception as e:
    print(f"❌ Authentication failed: {e}")


🔑 Authenticating user: amshahid@ncsu.edu
✅ Authentication successful!
✅ Authentication successful!
🔑 Token (first 20 chars): eyJ0eXAiOiJKV1QiLCJh...
📅 Token expiry: 1767892624.7769718


## 3. Get Dataset Metadata

Let's explore the dataset structure and understand what's available.

In [9]:
print("="*60)
print("📊 FETCHING DATASET METADATA")
print("="*60)

try:
    dataset = client.get_dataset(DATASET_UUID)
    
    print(f"\n📋 Dataset UUID: {DATASET_UUID}")
    print(f"📊 Total Records: {dataset.get('count', 0)}")
    print(f"🔢 Records in this page: {len(dataset.get('records', []))}")
    
    # Display first record as sample
    if dataset.get('records'):
        first_record = dataset['records'][0]
        print(f"\n📦 Sample Record:")
        print(f"  - Name: {first_record.get('record_name', 'N/A')}")
        print(f"  - UUID: {first_record.get('record_uuid', 'N/A')}")
        print(f"  - Fields: {len(first_record.get('fields', []))}")
        
except Exception as e:
    print(f"❌ Error fetching dataset: {e}")

📊 FETCHING DATASET METADATA
📋 Fetching dataset: 063c0d3d4bd183ab0dda87c544ae
✅ Retrieved 27 records

📋 Dataset UUID: 063c0d3d4bd183ab0dda87c544ae
📊 Total Records: 27
🔢 Records in this page: 27

📦 Sample Record:
  - Name: 772924
  - UUID: 1a64f214ae1e9c43aece0dfb7623
  - Fields: 7


## 4. Print Formatted Metadata

The client has a built-in method to print nicely formatted metadata.

In [10]:
# Use the built-in metadata printer
print("\n🎨 FORMATTED METADATA VIEW:")
client.print_metadata(DATASET_UUID)


🎨 FORMATTED METADATA VIEW:
📋 Fetching dataset: 063c0d3d4bd183ab0dda87c544ae
✅ Retrieved 27 records

📊 DATASET METADATA
Total Records: 27

📦 Record 1:
  - Name: 772924
  - UUID: 1a64f214ae1e9c43aece0dfb7623
  - Database UUID: 063c0d3d4bd183ab0dda87c544ae
  - Template UUID: 
  - Total Files: 1

📦 Record 2:
  - Name: 772925
  - UUID: 989bd5719310796f053c76e3ffa8
  - Database UUID: 063c0d3d4bd183ab0dda87c544ae
  - Template UUID: 
  - Total Files: 1

📦 Record 3:
  - Name: 772926
  - UUID: 666cd11f8b140c777c715dcfcf60
  - Database UUID: 063c0d3d4bd183ab0dda87c544ae
  - Template UUID: 
  - Total Files: 1

📦 Record 4:
  - Name: 772927
  - UUID: ce2a204f8ae5acd403d3c8bf6ada
  - Database UUID: 063c0d3d4bd183ab0dda87c544ae
  - Template UUID: 
  - Total Files: 1

📦 Record 5:
  - Name: 772962
  - UUID: 60862b03967fc4e8a73d57326b39
  - Database UUID: 063c0d3d4bd183ab0dda87c544ae
  - Template UUID: 
  - Total Files: 1

📦 Record 6:
  - Name: 773223
  - UUID: db4ad3151bee0ccb3d92ddb83ce9
  - Database 

## 5. Extract and Analyze Dataset Structure

Let's analyze the structure of records and fields in more detail.

In [11]:
# Analyze dataset structure
dataset = client.get_dataset(DATASET_UUID)

print("📊 DATASET STRUCTURE ANALYSIS")
print("="*50)

# Count different types of elements
total_records = dataset.get('count', 0)
total_files = 0
field_names = set()
file_types = set()

for record in dataset.get('records', []):
    # Collect field names
    for field in record.get('fields', []):
        field_names.add(field.get('field_name', 'Unknown'))
        
        # Count files and types
        for file_info in field.get('files', []):
            total_files += 1
            original_name = file_info.get('original_name', '')
            if '.' in original_name:
                extension = original_name.split('.')[-1].lower()
                file_types.add(extension)

print(f"\n📈 Summary:")
print(f"  - Total Records: {total_records}")
print(f"  - Total Files: {total_files}")
print(f"  - Unique Fields: {len(field_names)}")
print(f"  - File Types: {', '.join(sorted(file_types)) if file_types else 'None'}")
print(f"\n📝 Field Names Found:")
for field_name in sorted(field_names):
    print(f"  - {field_name}")

📋 Fetching dataset: 063c0d3d4bd183ab0dda87c544ae
✅ Retrieved 27 records
📊 DATASET STRUCTURE ANALYSIS

📈 Summary:
  - Total Records: 27
  - Total Files: 18
  - Unique Fields: 9
  - File Types: csv, tab, txt

📝 Field Names Found:
  - Class
  - Data File
  - Data Type
  - Notes
  - Sample Type
  - Source Citation
  - Source ID
  - Source Links
  - Subclass


## 6. Download All Files from Dataset

Download all files from the dataset into a local directory.

In [8]:
# Create downloads directory
output_dir = "odr_downloads"
print(f"📥 Downloading all files to: {output_dir}/")

try:
    client.extract_and_download_all_files(DATASET_UUID, output_dir=output_dir)
    
    # List downloaded files
    if os.path.exists(output_dir):
        files = list(Path(output_dir).glob('*'))
        print(f"\n📁 Downloaded {len(files)} files:")
        for file in files[:10]:  # Show first 10 files
            size = os.path.getsize(file)
            print(f"  - {file.name} ({size:,} bytes)")
        if len(files) > 10:
            print(f"  ... and {len(files) - 10} more files")
            
except Exception as e:
    print(f"❌ Error downloading files: {e}")

📥 Downloading all files to: odr_downloads/
📋 Fetching dataset: 063c0d3d4bd183ab0dda87c544ae
✅ Retrieved 5 records
📝 Metadata saved to: odr_downloads\063c0d3d4bd183ab0dda87c544ae_metadata.json

📦 Processing Record 1: 772924
  📄 Found file: ehlmann_2012.csv
📥 Downloading file: bee55da415556a603c45e65f9874
✅ Downloaded: ehlmann_2012.csv (782 bytes)

📦 Processing Record 2: 772925
  📄 Found file: kleine_2018.csv
📥 Downloading file: 57e366fa3001064ae2b3d247bf99
✅ Downloaded: kleine_2018.csv (1,835 bytes)

📦 Processing Record 3: 772926
  📄 Found file: RRUFF_Augite_R061108.txt
📥 Downloading file: 5854b539dd9725742d43f5019a3a
✅ Downloaded: RRUFF_Augite_R061108.txt (25,922 bytes)

📦 Processing Record 4: 772927
  📄 Found file: AC-PCP-001.tab
📥 Downloading file: 33618d62474cbf4703c60a063ad1
✅ Downloaded: AC-PCP-001.tab (8,143 bytes)

📦 Processing Record 5: 772962
  📄 Found file: test_upload.txt
📥 Downloading file: b488acdacd238cd13d7728922a41
✅ Downloaded: test_upload.txt (30 bytes)

✅ Download co

## 7. Download a Specific File

Test downloading a single file by its UUID.

In [9]:
# Download a specific file (you'll need to replace with an actual file UUID from your dataset)
test_file_uuid = "33618d62474cbf4703c60a063ad1"  # Replace with actual file UUID
output_path = "test_downloads"

print(f"📥 Attempting to download specific file: {test_file_uuid}")

try:
    result = client.download_file(
        file_uuid=test_file_uuid, 
        output_path=output_path,
        output_filename="test_file.txt"
    )
    
    if result:
        print(f"✅ File downloaded successfully to: {result}")
        # Check file exists
        if os.path.exists(result):
            size = os.path.getsize(result)
            print(f"📊 File size: {size:,} bytes")
    else:
        print("⚠️ File download returned None (might be forbidden)")
        
except Exception as e:
    print(f"❌ Error downloading file: {e}")

📥 Attempting to download specific file: 33618d62474cbf4703c60a063ad1
📥 Downloading file: 33618d62474cbf4703c60a063ad1
✅ Downloaded: test_file.txt (8,143 bytes)
✅ File downloaded successfully to: test_downloads\test_file.txt
📊 File size: 8,143 bytes


## 8. Get a Single Record by UUID

Fetch complete data for a specific record.

In [16]:
# Get a specific record (replace with actual record UUID from your dataset)
TARGET_RECORD_UUID = "420a340058dd57b5b162536b6f33"  # Replace with actual UUID

print(f"📦 Fetching record: {TARGET_RECORD_UUID}")

try:
    record = client.get_record(TARGET_RECORD_UUID)
    
    print(f"\n✅ Record fetched successfully!")
    print(f"📋 Record Details:")
    print(f"  - Name: {record.get('record_name', 'N/A')}")
    print(f"  - UUID: {record.get('record_uuid', 'N/A')}")
    print(f"  - Fields: {len(record.get('fields', []))}")
    
    # Show field values
    print(f"\n📝 Field Values:")
    for field in record.get('fields', []):
        field_name = field.get('field_name', 'Unknown')
        value = field.get('value', field.get('selected', 'N/A'))
        print(f"  - {field_name}: {value}")
        
    # Save record to file for inspection
    with open('sample_record.json', 'w') as f:
        json.dump(record, f, indent=2)
    print(f"\n💾 Full record saved to: sample_record.json")
    
except Exception as e:
    print(f"❌ Error fetching record: {e}")

📦 Fetching record: 420a340058dd57b5b162536b6f33
Fetching record: 420a340058dd57b5b162536b6f33

✅ Record fetched successfully!
📋 Record Details:
  - Name: 777390
  - UUID: 420a340058dd57b5b162536b6f33
  - Fields: 12

📝 Field Values:
  - Data File: N/A
  - Data Type: N/A
  - Source ID: Dick_1992
  - Class: N/A
  - Subclass: N/A
  - Sample Type: N/A
  - Source Links: Test
  - Source Citation: Test
  - Tags: N/A
  - Downloadable?: 1
  - Units: N/A
  - Methods: N/A

💾 Full record saved to: sample_record.json


Show the json file 

In [13]:
## show sample_record.json
print("\n📂 Sample Record JSON:")
try:
    with open('sample_record.json', 'r') as f:
        sample_record = json.load(f)
    print(json.dumps(sample_record, indent=2))
except FileNotFoundError:
    print("⚠️ sample_record.json not found. Please ensure the record was fetched successfully.")
    


📂 Sample Record JSON:
{
  "database_uuid": "063c0d3d4bd183ab0dda87c544ae",
  "internal_id": 772962,
  "record_name": "772962",
  "record_uuid": "60862b03967fc4e8a73d57326b39",
  "template_uuid": "",
  "metadata_for_uuid": "",
  "_record_metadata": {
    "_create_date": "2025-06-18 20:15:29",
    "_update_date": "2025-07-16 21:48:45",
    "_create_auth": "amshahid@ncsu.edu",
    "_public_date": "2200-01-01 00:00:00"
  },
  "fields": [
    {
      "field_name": "Data File",
      "field_uuid": "a65467babf8a1ac7e1d7319e3928",
      "template_field_uuid": null,
      "id": 8087,
      "files": [
        {
          "id": 215753,
          "file_uuid": "b488acdacd238cd13d7728922a41",
          "file_size": "30",
          "href": "https://www.odr.io/view/downloadfile/215753",
          "_file_metadata": {
            "_external_id": "",
            "_create_date": "2025-06-18 21:19:23",
            "_create_auth": "amshahid@ncsu.edu",
            "_public_date": "2200-01-01 00:00:00",
    

## 9. Modify and Update a Record

Demonstrate how to modify field values and push changes back to the server.

In [12]:
# Fetch a record to modify
print("🔄 RECORD MODIFICATION DEMO")
print("="*50)

try:
    # Get the record
    record = client.get_record(TARGET_RECORD_UUID)
    print(f"✅ Record fetched: {record.get('record_name')}")
    
    # Show current field values
    print("\n📝 Current Field Values:")
    for field in record.get('fields', []):
        if field.get('field_name') == 'Source':
            print(f"  - Source: {field.get('value', 'N/A')}")
    
    # Modify a field value
    print("\n✏️ Modifying 'Source' field...")
    client.set_field_value(record, field_name="Source", new_value="RRUFF (v2) - Modified")
    
    # Show modified value
    print("\n📝 Modified Field Values:")
    for field in record.get('fields', []):
        if field.get('field_name') == 'Source':
            print(f"  - Source: {field.get('value', 'N/A')}")
    
    # Push changes back to server
    print("\n📤 Pushing changes to server...")
    result = client.push_record(record)
    print("✅ Record updated successfully!")
    
except Exception as e:
    print(f"❌ Error updating record: {e}")

🔄 RECORD MODIFICATION DEMO
Fetching record: 60862b03967fc4e8a73d57326b39



                                    
        

                {
            "database_uuid": "063c0d3d4bd183ab0dda87c544ae",
            "internal_id": 772962,
            "record_name": "772962",               "record_uuid": "60862b03967fc4e8a73d57326b39",
                            "template_uuid": "",
                                        "metadata_for_uuid": "",
                                                        "_record_metadata": {
                                            "_create_date": "2025-06-18 20:15:29",
                                                                "_update_date": "2025-06-27 03:56:44",
                                                                "_create_auth": "amshahid@ncsu.edu",
                                                                "_public_date": "2200-01-01 00:00:00"
                                    },
                                             

Show updated record UUID by JSON 

In [14]:
## show updated record UUID by exporting to json then printing 
print("\n📂 Updated Record JSON:")
try:
    updated_record = client.get_record(TARGET_RECORD_UUID)
    with open('updated_record.json', 'w') as f:
        json.dump(updated_record, f, indent=2)
    print(json.dumps(updated_record, indent=2))
except Exception as e:
    print(f"❌ Error fetching updated record: {e}")


📂 Updated Record JSON:
Fetching record: 60862b03967fc4e8a73d57326b39



                                    
        

                {
            "database_uuid": "063c0d3d4bd183ab0dda87c544ae",
            "internal_id": 772962,
            "record_name": "772962",               "record_uuid": "60862b03967fc4e8a73d57326b39",
                            "template_uuid": "",
                                        "metadata_for_uuid": "",
                                                        "_record_metadata": {
                                            "_create_date": "2025-06-18 20:15:29",
                                                                "_update_date": "2025-07-16 21:48:45",
                                                                "_create_auth": "amshahid@ncsu.edu",
                                                                "_public_date": "2200-01-01 00:00:00"
                                    },
                                                

## 10. Create a New Record

Create a new record in the dataset.

In [15]:
# Create a new record
print("➕ CREATING NEW RECORD")
print("="*50)

try:
    # Create the record
    new_record = client.create_record(DATASET_UUID, user_email=USERNAME)
    
    print(f"✅ New record created!")
    print(f"📋 Record Details:")
    print(f"  - Name: {new_record.get('record_name', 'N/A')}")
    print(f"  - UUID: {new_record.get('record_uuid', 'N/A')}")
    print(f"  - Database UUID: {new_record.get('database_uuid', 'N/A')}")
    
    # Save new record UUID for later use
    new_record_uuid = new_record.get('record_uuid')
    print(f"\n💾 New record UUID saved: {new_record_uuid}")
    
except Exception as e:
    print(f"❌ Error creating record: {e}")

➕ CREATING NEW RECORD
📝 Creating new record in dataset: 063c0d3d4bd183ab0dda87c544ae
✅ Created new record: 773223 (UUID: db4ad3151bee0ccb3d92ddb83ce9)
✅ New record created!
📋 Record Details:
  - Name: 773223
  - UUID: db4ad3151bee0ccb3d92ddb83ce9
  - Database UUID: 063c0d3d4bd183ab0dda87c544ae

💾 New record UUID saved: db4ad3151bee0ccb3d92ddb83ce9


## 11. Upload a File to a Record

Upload a file and attach it to a record field.

In [ ]:
# Create a test file to upload
test_file_path = "test_upload.txt"
with open(test_file_path, 'w') as f:
    f.write("This is a test file for ODR upload.\n")
    f.write(f"Created at: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write("This file will be uploaded to the ODR system.")

print(f"📄 Created test file: {test_file_path}")

# To upload a file, you need:
# - record_uuid: The record to attach the file to
# - template_field_uuid: The field template UUID (get from metadata)
# - field_uuid: The specific field UUID (optional)

print("\n⚠️ Note: File upload requires knowing the template_field_uuid")
print("You can find this by examining the metadata of existing records with files.")

# Example upload (uncomment and modify with correct UUIDs):
"""
try:
    # You need to get these UUIDs from your dataset metadata
    record_uuid = new_record_uuid  # or use an existing record UUID
    template_field_uuid = "GET_THIS_FROM_METADATA"  # Replace with actual UUID
    field_uuid = ""  # Optional, can be empty
    
    upload_result = client.upload_file(
        file_path=test_file_path,
        record_uuid=record_uuid,
        dataset_uuid=DATASET_UUID,
        template_field_uuid=template_field_uuid,
        field_uuid=field_uuid,
        name="Test Upload File",
        user_email=USERNAME
    )
    
    print(f"✅ File uploaded successfully!")
    print(f"📋 Upload result: {json.dumps(upload_result, indent=2)}")
    
except Exception as e:
    print(f"❌ Error uploading file: {e}")
"""

# Clean up test file
if os.path.exists(test_file_path):
    os.remove(test_file_path)
    print(f"🧹 Cleaned up test file")

## 12. Advanced: Working with Nested Records

Some records may contain nested records. Let's explore how to work with them.

In [17]:
# Look for records with nested data
print("🔍 SEARCHING FOR NESTED RECORDS")
print("="*50)

dataset = client.get_dataset(DATASET_UUID)
records_with_nested = []

for record in dataset.get('records', []):
    nested = record.get('records', [])
    if nested:
        records_with_nested.append(record)
        print(f"\n📦 Record '{record.get('record_name')}' has {len(nested)} nested records:")
        for i, nested_record in enumerate(nested, 1):
            print(f"  └─ {i}. {nested_record.get('record_name', 'N/A')}")

if not records_with_nested:
    print("ℹ️ No nested records found in this dataset")
else:
    print(f"\n📊 Total records with nested data: {len(records_with_nested)}")

🔍 SEARCHING FOR NESTED RECORDS
📋 Fetching dataset: 063c0d3d4bd183ab0dda87c544ae
✅ Retrieved 6 records
ℹ️ No nested records found in this dataset


Further Development will be added here in terms of added functionality.

## Notes

- Remember to replace placeholder UUIDs with actual values from your dataset
- Store credentials securely (use environment variables in production)
- The API uses Bearer token authentication that expires; the client handles refresh automatically
- File uploads require knowing the correct template_field_uuid
- Some operations may require specific permissions on the ODR system